# Supervised and self-supervised, block by block

A demo: the same photos through encoders that differ in their training, supervised on ImageNet labels or
self-supervised (DINO), side by side, for ViT-S/16 and ResNet50. For each model: its tokens at every block,
from the input of block 1 (`b0`) to the last, shown with UMAP and coloured by the model's own concepts at its
last block, with the chosen photo's explanation at every block.

Models have different numbers of blocks (13 frames for a ViT, 5 for a ResNet), so the rows advance by
**relative depth**: all start at their input and reach their last block together.

**Needs:** `pip install "nave-xai[animation]"`. **Downloads:** each model's weights once (about 90 MB each).

In [ ]:
import sys
if "google.colab" in sys.modules:       # on Google Colab: install what this notebook needs
    get_ipython().run_line_magic("pip", 'install -q "nave-xai[animation]"')
# elsewhere: pip install "nave-xai[animation]"
import numpy as np

MODELS = [("vits16", "imagenet"), ("vits16", "dino"), ("resnet50", "imagenet"), ("resnet50", "dino")]
K = 8
N_IMAGES = 12
IMAGE = 2                                  # the photo shown (2: the cat); re-run only the drawing cells to change it
UMAP_EPOCHS = 200
SEED = 0


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display


def match_colours(labels, reference, K):
    """Give every concept of `labels` the colour index of the reference concept it overlaps most
    (one-to-one, maximising the total overlap), so colours stay stable from frame to frame."""
    overlap = np.zeros((K, K))
    np.add.at(overlap, (labels.ravel(), reference.ravel()), 1)
    try:
        from scipy.optimize import linear_sum_assignment
        rows, cols = linear_sum_assignment(-overlap)
        mapping = dict(zip(rows, cols))
    except ImportError:                                     # greedy, without scipy
        mapping, used = {}, set()
        for a, b in sorted(np.ndindex(K, K), key=lambda ab: -overlap[ab]):
            if a not in mapping and b not in used:
                mapping[a] = b
                used.add(b)
    return np.vectorize(lambda a: mapping.get(a, a))(labels)


def knn_agreement(features, targets, k=5, n=1500, seed=0):
    """How well a point's k nearest neighbours (cosine) predict its target: leave-one-out accuracy."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(features), min(n, len(features)), replace=False)
    F = features[idx] / np.linalg.norm(features[idx], axis=1, keepdims=True)
    t = targets[idx]
    sim = F @ F.T
    np.fill_diagonal(sim, -np.inf)
    nearest = np.argsort(-sim, axis=1)[:, :k]
    votes = np.array([np.bincount(t[row], minlength=t.max() + 1).argmax() for row in nearest])
    return float((votes == t).mean())


def normalise_embedding(e):
    e = e - np.median(e, axis=0)
    return e / np.percentile(np.abs(e), 98)


def token_labels(maps, centres):
    """The concept at every patch centre of every image: one label per token."""
    return np.concatenate([m[np.ix_(centres, centres)].ravel() for m in maps])


def match_to(labels, reference, K):
    """Rename `labels` so each concept takes the id of the reference concept it overlaps most."""
    return match_colours(labels, reference, K)


def block_names(backbone):
    """The input of block 1 ("b0"), then every block."""
    return ["b0"] + list(backbone.blocks)


def pool_to(m, gh, gw):
    """Average a (C, h, w) map onto a gh x gw grid (each cell: the mean of the cells it covers)."""
    C, h, w = m.shape
    if (h, w) == (gh, gw):
        return m
    if h % gh == 0 and w % gw == 0:
        return m.reshape(C, gh, h // gh, gw, w // gw).mean((2, 4))
    from skimage.transform import resize
    return resize(m, (C, gh, gw), order=1, anti_aliasing=True, preserve_range=True)


def block_tokens(backbone, images, names):
    """For every block: the tokens of all images on the deepest block's grid, scaled to length 1.
    Returns (tokens per block, (grid height, grid width), cell size in pixels)."""
    mean, std = np.array(backbone.mean, np.float32), np.array(backbone.std, np.float32)
    per_block = {n: [] for n in names}
    grid = None
    for img in images:
        x = ((img / 255.0 - mean) / std).astype(np.float32).transpose(2, 0, 1)[None]
        out = backbone(x, names)
        if grid is None:
            grid = out[names[-1]].shape[2:]
        for n in names:
            t = pool_to(out[n][0], *grid)
            per_block[n].append(t.reshape(t.shape[0], -1).T)
    tokens = [np.concatenate(per_block[n]) for n in names]
    tokens = [t / (np.linalg.norm(t, axis=1, keepdims=True) + 1e-12) for t in tokens]
    return tokens, tuple(grid), images[0].shape[0] // grid[0]


def cell_centres(size, cell):
    return np.arange(cell // 2, size, cell)


def aligned_umap(features, epochs, window=2, regularisation=0.02):
    """UMAP of a sequence of point sets (the same points), each frame kept close to its neighbours."""
    a = umap.AlignedUMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs, alignment_regularisation=regularisation,
                         alignment_window_size=window)
    a.fit(features, relations=[{i: i for i in range(len(features[0]))}] * (len(features) - 1))
    return [normalise_embedding(e) for e in a.embeddings_]


def last_reference_umap(features, epochs):
    """UMAP of every frame, each starting from the last frame's layout."""
    last = umap.UMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs).fit_transform(features[-1])
    out = [umap.UMAP(n_neighbors=15, min_dist=0.2, n_epochs=epochs, init=last).fit_transform(f) for f in features[:-1]]
    return [normalise_embedding(e) for e in out + [last]]


def renamed_like(labels, reference, K):
    """A renaming of concept ids so each takes the id of the reference concept it overlaps most."""
    from scipy.optimize import linear_sum_assignment
    overlap = np.zeros((K, K))
    np.add.at(overlap, (labels, reference), 1)
    r, c = linear_sum_assignment(-overlap)
    rename = np.arange(K)
    rename[r] = c
    return rename

import warnings
import umap

warnings.filterwarnings("ignore", module="umap")               # UMAP's notices about threads and seeds
warnings.filterwarnings("ignore", category=FutureWarning)      # scikit-learn's, inside UMAP


In [ ]:
from skimage import data, transform

names = ["astronaut", "coffee", "chelsea", "rocket", "immunohistochemistry", "hubble_deep_field",
         "retina", "camera", "coins", "horse", "moon", "page"][:N_IMAGES]

def load(name):
    img = getattr(data, name)()
    img = np.repeat(img[..., None], 3, axis=2) if img.ndim == 2 else img[..., :3]
    return (transform.resize(img, (224, 224), anti_aliasing=False) * 255).astype(np.uint8)

images = [load(n) for n in names]
fig, axes = plt.subplots(1, len(images), figsize=(1.3 * len(images), 1.5))
for ax, im, n in zip(np.atleast_1d(axes), images, names):
    ax.imshow(im); ax.set_title(n, fontsize=7); ax.axis("off")
# your own photos: see the commented cell in 01_encoder_blocks.ipynb

## One journey per model

For each model: its tokens at every block (ResNets: averaged onto the last block's grid), its own concepts
at the last block, AlignedUMAP over the blocks, and NAVE fitted on each block alone, for every photo, colours
matched to the last block's.

In [ ]:
from nave_xai import NAVEImage

def journey(model, training):
    final = NAVEImage(model, training, k=K, seed=SEED)
    bb = final.backbone
    names_b = block_names(bb)
    final = NAVEImage(bb, layers=[names_b[-1]], k=K, seed=SEED).fit(images)
    tokens, grid, cell = block_tokens(bb, images, names_b)
    centres = cell_centres(224, cell)
    concept = token_labels([e.explanation for e in final.predict(images)], centres)
    palette = final.palette / 255
    views = []
    for name in names_b:
        nave = NAVEImage(bb, layers=[name], k=K, seed=SEED).fit(images)
        exps = nave.predict(images)
        rename = renamed_like(token_labels([e.explanation for e in exps], centres), concept, K)
        views.append([(palette[rename[e.explanation]] * 255).astype(np.uint8) for e in exps])
    return dict(name=f"{model}, {training}", names=names_b, emb=aligned_umap(tokens, UMAP_EPOCHS),
                colours=palette[concept], G=grid[0] * grid[1], views=views,
                agreement=[knn_agreement(t, concept, seed=SEED) for t in tokens])

journeys = [journey(m, t) for m, t in MODELS]
print("ready:", [j["name"] for j in journeys])

## The animation

One row per model; the horizontal axis of the curves is relative depth (0: input of block 1, 1: last block).

In [ ]:
def animate_models(frames=60):
    n = len(journeys)
    plt.rcParams["animation.embed_limit"] = 100
    fig = plt.figure(figsize=(9.5, 2.8 * n + 1.5), dpi=60)
    grid_ = fig.add_gridspec(n + 1, 3, height_ratios=[3] * n + [1.1], hspace=0.35)
    ax_in = fig.add_subplot(grid_[:n, 0]); ax_in.imshow(images[IMAGE]); ax_in.axis("off")
    ax_in.set_title(f"input: {names[IMAGE]}", fontsize=9)
    curve = fig.add_subplot(grid_[n, :])
    arts = []
    for r, j in enumerate(journeys):
        mine = np.arange(IMAGE * j["G"], (IMAGE + 1) * j["G"])
        a_u, a_e = fig.add_subplot(grid_[r, 1]), fig.add_subplot(grid_[r, 2])
        for a in (a_u, a_e):
            a.set_xticks([]); a.set_yticks([])
        a_u.set_xlim(-1.4, 1.4); a_u.set_ylim(-1.4, 1.4); a_u.set_ylabel(j["name"], fontsize=9)
        every = a_u.scatter(*j["emb"][0].T, c=j["colours"], s=4, linewidths=0)
        sel = a_u.scatter(*j["emb"][0][mine].T, c=j["colours"][mine], s=16, edgecolors="black", linewidths=0.4)
        im = a_e.imshow(j["views"][0][IMAGE])
        if r == 0:
            a_u.set_title("UMAP of all tokens", fontsize=9); a_e.set_title("NAVE explanation", fontsize=9)
        depth = np.linspace(0, 1, len(j["names"]))
        curve.plot(depth, j["agreement"], marker="o", ms=3, label=j["name"])
        arts.append((j, every, sel, im, mine))
    curve.set_ylim(0, 1); curve.set_xlim(-0.02, 1.02); curve.legend(fontsize=7)
    curve.set_xlabel("relative depth (0: input of block 1, 1: last block)", fontsize=8)
    marker = curve.axvline(0, color="black", lw=1)
    head = fig.suptitle("", fontsize=11)

    def frame(f):
        d = f / (frames - 1)
        texts = []
        for j, every, sel, im, mine in arts:
            t = d * (len(j["names"]) - 1); i0, w = int(t), t - int(t); i1 = min(i0 + 1, len(j["names"]) - 1)
            pos = (1 - w) * j["emb"][i0] + w * j["emb"][i1]
            every.set_offsets(pos); sel.set_offsets(pos[mine])
            b = int(round(t)); im.set_data(j["views"][b][IMAGE]); texts.append(j["names"][b])
        marker.set_xdata([d, d]); head.set_text("  |  ".join(f"{j['name']}: {t_}" for (j, *_), t_ in zip(arts, texts)))
        return [marker, head]

    return animation.FuncAnimation(fig, frame, frames=frames, interval=120, blit=False)


anim = animate_models()
display(HTML(anim.to_jshtml()))
anim.save("supervised_vs_selfsupervised.gif", writer=animation.PillowWriter(fps=8), dpi=70)
plt.close()

## Things to look at

- which model separates its final concepts earlier, in relative depth;
- ViT against ResNet with the same training, and supervised against self-supervised with the same architecture;
- what each model's concepts cover in the photos: objects, parts, textures.

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```